
# DeepFace 기반 얼굴 유사도 비교 + DB 등록/인증 서비스

이 노트북은 기존 `09_03이미지분석_사진유사도비교_서비스-deepface.ipynb`에
`09_02mediapipe로 사용자 안면인증 서비스 만들기.ipynb`의 **DB 저장 / 조회 / 삭제 / 인증 로직**을 합친 버전입니다.

## 포함 기능
- 두 이미지 직접 비교 (`DeepFace.verify`)
- 얼굴 임베딩 추출 후 MySQL DB 저장
- DB에 저장된 사용자들과 실시간 비교 인증
- 등록 사용자 목록 조회 및 삭제

## 변경 포인트
- 최신 DeepFace 사용 방식에 맞춰 `img1_path=...`, `img2_path=...` 형태로 호출
- 이미지 파일 저장 없이 NumPy 배열을 직접 전달
- DB에는 DeepFace 임베딩 벡터(float32)를 저장
- 인증은 등록된 모든 사용자 임베딩과 비교하여 최고 유사 사용자를 반환


In [1]:

# 필요 시 설치
# %pip install -U deepface tf-keras "gradio>=5,<6" "sqlalchemy>=2" pymysql opencv-python


In [2]:

import time
import cv2
import numpy as np
import gradio as gr

from deepface import DeepFace
from sqlalchemy import create_engine, text


/home/ksjin/miniforge3/envs/deep/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-24 10:52:57.090697: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-24 10:52:58.001349: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-24 10:53:01.054327: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may se

26-03-24 10:53:02 - Directory /home/ksjin/.deepface has been created
26-03-24 10:53:02 - Directory /home/ksjin/.deepface/weights has been created


In [3]:

print('gradio version   :', gr.__version__)


gradio version   : 6.9.0


In [4]:

# =========================
# DeepFace 설정
# =========================
MODELS = [
    "ArcFace",
    "SFace",
    "Facenet512",
    "Facenet",
    "GhostFaceNet",
    "Buffalo_L",
    "VGG-Face",
    "Dlib",
    "OpenFace",
    "DeepFace",
    "DeepID",
]

DETECTOR_BACKENDS = [
    "retinaface",
    "opencv",
]

RECOMMENDED_METRIC = {
    "ArcFace": "cosine",
    "SFace": "cosine",
    "Facenet512": "euclidean_l2",
    "Facenet": "euclidean_l2",
    "GhostFaceNet": "cosine",
    "Buffalo_L": "cosine",
    "VGG-Face": "euclidean",
    "Dlib": "euclidean_l2",
    "OpenFace": "euclidean_l2",
    "DeepFace": "euclidean",
    "DeepID": "euclidean",
}


In [5]:

# =========================
# MySQL 접속 정보
# =========================
DB_HOST = "172.16.24.149"
DB_PORT = 3306
DB_USER = "face_auth"
DB_PASSWORD = "1234"
DB_NAME = "face_auth_db"

DB_URL_SERVER = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/?charset=utf8mb4"
DB_URL_DB = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4"

server_engine = create_engine(
    DB_URL_SERVER,
    pool_pre_ping=True,
    future=True,
)

engine = create_engine(
    DB_URL_DB,
    pool_pre_ping=True,
    future=True,
)


In [6]:

def init_database():
    with server_engine.begin() as conn:
        conn.execute(text(
            f"CREATE DATABASE IF NOT EXISTS {DB_NAME} "
            f"CHARACTER SET utf8mb4 COLLATE utf8mb4_general_ci"
        ))

    with engine.begin() as conn:
        conn.execute(text("""
            CREATE TABLE IF NOT EXISTS deepface_users (
                id INT AUTO_INCREMENT PRIMARY KEY,
                user_name VARCHAR(100) NOT NULL UNIQUE,
                model_name VARCHAR(50) NOT NULL,
                detector_backend VARCHAR(50) NOT NULL,
                distance_metric VARCHAR(50) NOT NULL,
                embedding_vector LONGBLOB NOT NULL,
                vector_dim INT NOT NULL,
                created_at DATETIME NOT NULL,
                updated_at DATETIME NOT NULL
            )
        """))


In [7]:

def save_face_to_db(user_name: str, model: str, detector_backend: str, distance_metric: str, embedding_vector: np.ndarray):
    now_str = time.strftime("%Y-%m-%d %H:%M:%S")
    vec = np.asarray(embedding_vector, dtype=np.float32)
    vec_bytes = vec.tobytes()
    vec_dim = int(vec.shape[0])

    with engine.begin() as conn:
        row = conn.execute(
            text("SELECT id FROM deepface_users WHERE user_name = :user_name"),
            {"user_name": user_name}
        ).fetchone()

        if row is None:
            conn.execute(text("""
                INSERT INTO deepface_users (
                    user_name, model_name, detector_backend, distance_metric,
                    embedding_vector, vector_dim, created_at, updated_at
                )
                VALUES (
                    :user_name, :model_name, :detector_backend, :distance_metric,
                    :embedding_vector, :vector_dim, :created_at, :updated_at
                )
            """), {
                "user_name": user_name,
                "model_name": model,
                "detector_backend": detector_backend,
                "distance_metric": distance_metric,
                "embedding_vector": vec_bytes,
                "vector_dim": vec_dim,
                "created_at": now_str,
                "updated_at": now_str,
            })
        else:
            conn.execute(text("""
                UPDATE deepface_users
                SET model_name = :model_name,
                    detector_backend = :detector_backend,
                    distance_metric = :distance_metric,
                    embedding_vector = :embedding_vector,
                    vector_dim = :vector_dim,
                    updated_at = :updated_at
                WHERE user_name = :user_name
            """), {
                "user_name": user_name,
                "model_name": model,
                "detector_backend": detector_backend,
                "distance_metric": distance_metric,
                "embedding_vector": vec_bytes,
                "vector_dim": vec_dim,
                "updated_at": now_str,
            })


def load_all_faces_from_db():
    users = []
    with engine.begin() as conn:
        rows = conn.execute(text("""
            SELECT id, user_name, model_name, detector_backend, distance_metric,
                   embedding_vector, vector_dim, created_at, updated_at
            FROM deepface_users
            ORDER BY id ASC
        """)).fetchall()

    for row in rows:
        vec = np.frombuffer(row.embedding_vector, dtype=np.float32)
        if len(vec) != row.vector_dim:
            continue
        users.append({
            "id": row.id,
            "user_name": row.user_name,
            "model_name": row.model_name,
            "detector_backend": row.detector_backend,
            "distance_metric": row.distance_metric,
            "embedding_vector": vec,
            "vector_dim": row.vector_dim,
            "created_at": str(row.created_at),
            "updated_at": str(row.updated_at),
        })
    return users


def delete_user_from_db(user_name: str):
    with engine.begin() as conn:
        result = conn.execute(
            text("DELETE FROM deepface_users WHERE user_name = :user_name"),
            {"user_name": user_name}
        )
        return result.rowcount


def get_registered_users_text():
    users = load_all_faces_from_db()
    if not users:
        return "등록된 사용자가 없습니다."

    lines = []
    for u in users:
        lines.append(
            f"[id={u['id']}] {u['user_name']} | model={u['model_name']} | detector={u['detector_backend']} | metric={u['distance_metric']} | updated_at={u['updated_at']}"
        )
    return "".join(lines)


def safe_get_registered_users_text():
    try:
        return get_registered_users_text()
    except Exception as e:
        return f"사용자 목록 조회 실패: {e}"


In [8]:

def ensure_bgr_uint8(img):
    if img is None:
        return None

    if not isinstance(img, np.ndarray):
        img = np.array(img)

    if img.dtype != np.uint8:
        if np.issubdtype(img.dtype, np.floating):
            if img.max() <= 1.0:
                img = (img * 255).clip(0, 255).astype(np.uint8)
            else:
                img = img.clip(0, 255).astype(np.uint8)
        else:
            img = img.astype(np.uint8)

    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    elif img.ndim == 3 and img.shape[2] == 1:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    elif img.ndim == 3 and img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_RGBA2BGR)
    elif img.ndim == 3 and img.shape[2] == 3:
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    else:
        raise ValueError(f"지원하지 않는 이미지 shape: {img.shape}")

    return img


def convert_distance_to_similarity(distance, threshold, verified):
    if verified:
        ratio = max(0, 1 - (distance / max(threshold, 1e-8)))
        return round(90 + ratio * 10, 2)
    ratio = max(0, 1 - (distance / max(threshold * 2, 1e-8)))
    return round(ratio * 60, 2)


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom < 1e-8:
        return 0.0
    return float(np.dot(a, b) / denom)


def euclidean_distance(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.linalg.norm(a - b))


In [9]:

def extract_embedding(image, model, detector_backend, enforce_detection=True, align=True):
    img = ensure_bgr_uint8(image)
    reps = DeepFace.represent(
        img_path=img,
        model_name=model,
        detector_backend=detector_backend,
        enforce_detection=enforce_detection,
        align=align,
        normalization="base",
    )

    if not reps:
        raise ValueError("얼굴 임베딩을 추출하지 못했습니다.")

    rep = reps[0]
    embedding = np.asarray(rep["embedding"], dtype=np.float32)
    facial_area = rep.get("facial_area")
    return embedding, facial_area


In [10]:

def compare_two_faces(image1, image2, model, detector_backend, enforce_detection, align):
    if image1 is None or image2 is None:
        return "두 이미지를 모두 업로드하거나 촬영해 주세요."

    try:
        img1 = ensure_bgr_uint8(image1)
        img2 = ensure_bgr_uint8(image2)
        metric = RECOMMENDED_METRIC.get(model, "cosine")

        faces1 = DeepFace.extract_faces(
            img_path=img1,
            detector_backend=detector_backend,
            enforce_detection=enforce_detection,
            align=align,
        )
        faces2 = DeepFace.extract_faces(
            img_path=img2,
            detector_backend=detector_backend,
            enforce_detection=enforce_detection,
            align=align,
        )

        result = DeepFace.verify(
            img1_path=img1,
            img2_path=img2,
            model_name=model,
            detector_backend=detector_backend,
            distance_metric=metric,
            enforce_detection=enforce_detection,
            align=align,
            silent=True,
        )

        distance = float(result.get("distance", 0.0))
        threshold = float(result.get("threshold", 0.0) or 0.5)
        verified = bool(result.get("verified", False))
        similarity_score = convert_distance_to_similarity(distance, threshold, verified)

        lines = [
            f"모델: {model}\n",
            f"Detector: {detector_backend}\n",
            f"img1 검출 얼굴 수: {len(faces1)}",
            f"img2 검출 얼굴 수: {len(faces2)}\n",
            f"거리 기준(metric): {metric}\n",
            f"distance: {distance:.6f}\n",
            f"threshold: {threshold:.6f}\n\n",
            f"유사도(가공값): {similarity_score:.2f}%\n",
            f"판정: {'일치' if verified else '불일치'}",
        ]
        return "".join(lines)
    except Exception as e:
        return f"오류 발생: {type(e).__name__}: {e}"


In [11]:

def enroll_face(image, user_name, model, detector_backend, enforce_detection, align):
    user_name = (user_name or "").strip()
    if not user_name:
        return image, "사용자 이름을 입력하세요.", safe_get_registered_users_text()
    if image is None:
        return None, "등록할 얼굴 이미지를 입력하세요.", safe_get_registered_users_text()

    try:
        metric = RECOMMENDED_METRIC.get(model, "cosine")
        embedding, facial_area = extract_embedding(
            image=image,
            model=model,
            detector_backend=detector_backend,
            enforce_detection=enforce_detection,
            align=align,
        )
        save_face_to_db(user_name, model, detector_backend, metric, embedding)

        msg_lines = [
            f"등록 완료: {user_name}",
            f"model={model}",
            f"detector={detector_backend}",
            f"metric={metric}",
            f"embedding_dim={embedding.shape[0]}",
        ]
        if facial_area is not None:
            msg_lines.append(f"facial_area={facial_area}")
        return image, "".join(msg_lines), safe_get_registered_users_text()
    except Exception as e:
        return image, f"등록 실패: {type(e).__name__}: {e}", safe_get_registered_users_text()


def verify_face(image, model, detector_backend, enforce_detection, align, cosine_threshold, euclidean_threshold):
    if image is None:
        return None, "인증할 얼굴 이미지를 입력하세요."

    users = load_all_faces_from_db()
    if not users:
        return image, "DB에 등록된 사용자가 없습니다."

    try:
        probe_embedding, _ = extract_embedding(
            image=image,
            model=model,
            detector_backend=detector_backend,
            enforce_detection=enforce_detection,
            align=align,
        )
    except Exception as e:
        return image, f"인증 실패: {type(e).__name__}: {e}"

    best_user = None
    best_cos = -1.0
    best_dist = float('inf')

    for u in users:
        db_vec = u["embedding_vector"]
        if db_vec.shape != probe_embedding.shape:
            continue

        cos = cosine_similarity(db_vec, probe_embedding)
        dist = euclidean_distance(db_vec, probe_embedding)

        if cos > best_cos:
            best_cos = cos
            best_dist = dist
            best_user = u

    if best_user is None:
        return image, "비교 가능한 등록 데이터가 없습니다. 모델이 다른 사용자만 등록되어 있을 수 있습니다."

    is_match = (best_cos >= cosine_threshold) or (best_dist <= euclidean_threshold)

    result_text = (
        f"인증 {'성공' if is_match else '실패'}\t"
        f"- 가장 유사한 사용자: {best_user['user_name']}\n\n"
        f"- 등록 모델: {best_user['model_name']}\n"
        f"- 등록 detector: {best_user['detector_backend']}\n"
        f"- Cosine similarity: {best_cos:.6f}\n"
        f"- Euclidean distance: {best_dist:.6f}\n"
        f"- Cosine threshold: {cosine_threshold:.6f}\n"
        f"- Euclidean threshold: {euclidean_threshold:.6f}\n"
    )
    return image, result_text


def delete_user(user_name):
    user_name = (user_name or "").strip()
    if not user_name:
        return "삭제할 사용자 이름을 입력하세요.", safe_get_registered_users_text()

    try:
        deleted_count = delete_user_from_db(user_name)
        if deleted_count > 0:
            return f"삭제 완료: {user_name}", safe_get_registered_users_text()
        return f"삭제 대상 없음: {user_name}", safe_get_registered_users_text()
    except Exception as e:
        return f"삭제 실패: {e}", safe_get_registered_users_text()


def refresh_users():
    return safe_get_registered_users_text()


In [12]:

try:
    init_database()
    db_status_text = "MySQL 연결 및 DB 초기화 성공"
except Exception as e:
    db_status_text = f"MySQL 초기화 실패: {e}"

print(db_status_text)


MySQL 연결 및 DB 초기화 성공


In [13]:

DESCRIPTION = """
# DeepFace 얼굴 유사도 비교 + DB 등록/인증

동작 순서
1. 이미지 비교 탭에서 두 얼굴을 직접 비교
2. 얼굴 등록 탭에서 사용자 이름과 얼굴 이미지를 입력해 DB 저장
3. 얼굴 인증 탭에서 입력 얼굴과 DB 등록 얼굴들을 비교
4. 사용자 삭제 탭에서 등록된 사용자 삭제
"""

with gr.Blocks(title="DeepFace 얼굴 등록/인증", theme=gr.themes.Soft()) as app:
    gr.Markdown(DESCRIPTION)

    db_status = gr.Textbox(
        label="DB 상태",
        value=db_status_text,
        interactive=False,
    )

    user_list_box = gr.Textbox(
        label="등록된 사용자 목록",
        value=safe_get_registered_users_text() if "성공" in db_status_text else "DB 연결 실패 상태",
        lines=8,
        interactive=False,
    )

    refresh_btn = gr.Button("사용자 목록 새로고침")

    with gr.Tab("두 이미지 비교"):
        with gr.Row():
            with gr.Column():
                compare_image1 = gr.Image(
                    label="첫 번째 얼굴",
                    type="numpy",
                    image_mode="RGB",
                    sources=["upload", "webcam", "clipboard"],
                )
            with gr.Column():
                compare_image2 = gr.Image(
                    label="두 번째 얼굴",
                    type="numpy",
                    image_mode="RGB",
                    sources=["upload", "webcam", "clipboard"],
                )

        with gr.Row():
            compare_model = gr.Dropdown(label="모델 선택", choices=MODELS, value="ArcFace")
            compare_detector = gr.Dropdown(label="얼굴 검출기 선택", choices=DETECTOR_BACKENDS, value="retinaface")

        with gr.Row():
            compare_enforce = gr.Checkbox(label="얼굴 미검출 시 오류 처리", value=True)
            compare_align = gr.Checkbox(label="얼굴 정렬(alignment) 사용", value=True)

        compare_output = gr.Textbox(label="비교 결과", lines=10)
        compare_btn = gr.Button("유사도 비교", variant="primary")

    with gr.Tab("얼굴 등록"):
        with gr.Row():
            enroll_input = gr.Image(
                label="등록용 얼굴 이미지",
                sources=["upload", "webcam", "clipboard"],
                type="numpy",
                image_mode="RGB",
            )
            enroll_output = gr.Image(label="입력 이미지", type="numpy")

        enroll_name = gr.Textbox(label="사용자 이름", placeholder="예: haram")

        with gr.Row():
            enroll_model = gr.Dropdown(label="모델 선택", choices=MODELS, value="ArcFace")
            enroll_detector = gr.Dropdown(label="얼굴 검출기 선택", choices=DETECTOR_BACKENDS, value="retinaface")

        with gr.Row():
            enroll_enforce = gr.Checkbox(label="얼굴 미검출 시 오류 처리", value=True)
            enroll_align = gr.Checkbox(label="얼굴 정렬(alignment) 사용", value=True)

        enroll_msg = gr.Textbox(label="등록 결과", lines=6)
        enroll_btn = gr.Button("얼굴 등록", variant="primary")

    with gr.Tab("얼굴 인증"):
        with gr.Row():
            verify_input = gr.Image(
                label="인증용 얼굴 이미지",
                sources=["upload", "webcam", "clipboard"],
                type="numpy",
                image_mode="RGB",
            )
            verify_output = gr.Image(label="입력 이미지", type="numpy")

        with gr.Row():
            verify_model = gr.Dropdown(label="모델 선택", choices=MODELS, value="ArcFace")
            verify_detector = gr.Dropdown(label="얼굴 검출기 선택", choices=DETECTOR_BACKENDS, value="retinaface")

        with gr.Row():
            verify_enforce = gr.Checkbox(label="얼굴 미검출 시 오류 처리", value=True)
            verify_align = gr.Checkbox(label="얼굴 정렬(alignment) 사용", value=True)

        cosine_threshold = gr.Slider(
            minimum=0.0,
            maximum=1.0,
            value=0.68,
            step=0.001,
            label="Cosine threshold",
        )
        euclidean_threshold = gr.Slider(
            minimum=0.0,
            maximum=25.0,
            value=4.0,
            step=0.01,
            label="Euclidean threshold",
        )

        verify_msg = gr.Textbox(label="인증 결과", lines=8)
        verify_btn = gr.Button("얼굴 인증", variant="primary")

    with gr.Tab("사용자 삭제"):
        delete_name = gr.Textbox(label="삭제할 사용자 이름")
        delete_msg = gr.Textbox(label="삭제 결과", lines=2)
        delete_btn = gr.Button("사용자 삭제", variant="stop")

    compare_btn.click(
        fn=compare_two_faces,
        inputs=[compare_image1, compare_image2, compare_model, compare_detector, compare_enforce, compare_align],
        outputs=[compare_output],
    )

    enroll_btn.click(
        fn=enroll_face,
        inputs=[enroll_input, enroll_name, enroll_model, enroll_detector, enroll_enforce, enroll_align],
        outputs=[enroll_output, enroll_msg, user_list_box],
    )

    verify_btn.click(
        fn=verify_face,
        inputs=[verify_input, verify_model, verify_detector, verify_enforce, verify_align, cosine_threshold, euclidean_threshold],
        outputs=[verify_output, verify_msg],
    )

    delete_btn.click(
        fn=delete_user,
        inputs=[delete_name],
        outputs=[delete_msg, user_list_box],
    )

    refresh_btn.click(
        fn=refresh_users,
        inputs=[],
        outputs=[user_list_box],
    )


In [14]:

try:
    app.close()
except Exception:
    pass

app.launch(
    server_name="127.0.0.1",
    server_port=7866,
    inline=False,
    inbrowser=True,
    share=False,
    prevent_thread_lock=True,
    show_error=True,
   
)


* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


2026-03-24 10:54:52.631266: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1774317292.631332    1680 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1347 MB memory:  -> device: 0, name: NVIDIA GeForce MX450, pci bus id: 0000:01:00.0, compute capability: 7.5


26-03-24 10:54:55 - retinaface.h5 will be downloaded from the url https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5
To: /home/ksjin/.deepface/weights/retinaface.h5
100%|████████████████████████████████████████████████████████████████████| 119M/119M [00:04<00:00, 28.6MB/s]
2026-03-24 10:55:03.664391: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 92000
2026-03-24 10:55:08.539963: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.09GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2026-03-24 10:55:08.964696: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.09GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were avai

26-03-24 10:55:19 - 🔗 arcface_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/arcface_weights.h5 to /home/ksjin/.deepface/weights/arcface_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/arcface_weights.h5
To: /home/ksjin/.deepface/weights/arcface_weights.h5
100%|████████████████████████████████████████████████████████████████████| 137M/137M [00:07<00:00, 18.2MB/s]
2026-03-24 10:55:30.722940: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.08GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2026-03-24 10:55:30.853597: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.08GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.


In [15]:

# 종료가 필요할 때 실행
# app.close()
